In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set.")

PYTORCH_CUDA_ALLOC_CONF set.


In [2]:
import sys
import subprocess

packages = [
    "langchain==1.3.9",
    "langchain-core==1.4.7",
    "langchain-community==0.4.2",
    "langchain-huggingface==1.2.2",
    "langchain-chroma==1.1.0",
    "langchain-text-splitters==1.1.2",
    "sentence-transformers==3.0.1",
    "chromadb",
    "pymupdf",
    "pdfplumber",
    "rank-bm25",
    "bitsandbytes",
    "accelerate",
    "scikit-learn",
    "bert-score",
]

subprocess.run(["pip", "install", "-q", "--no-cache-dir"] + packages, check=True)

#Auto-restart kernel so all installs are visible immediately

#print("✅ Packages installed — restarting kernel...")

#import IPython

#IPython.Application.instance().kernel.do_shutdown(restart=True)

CompletedProcess(args=['pip', 'install', '-q', '--no-cache-dir', 'langchain==1.3.9', 'langchain-core==1.4.7', 'langchain-community==0.4.2', 'langchain-huggingface==1.2.2', 'langchain-chroma==1.1.0', 'langchain-text-splitters==1.1.2', 'sentence-transformers==3.0.1', 'chromadb', 'pymupdf', 'pdfplumber', 'rank-bm25', 'bitsandbytes', 'accelerate', 'scikit-learn', 'bert-score'], returncode=0)

In [3]:
from pathlib import Path
from langchain_core.documents import Document
import fitz        
import pdfplumber
import numpy as np  
pdf_dir = "/kaggle/input/datasets/shivammusk/sec-filings/SEC Filings"
pdf_files = list(Path(pdf_dir).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")

all_documents = []

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name}")
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text").strip()

        if len(text) < 30:
            continue

        # Text block
        all_documents.append(Document(
            page_content=text,
            metadata={
                "source": str(pdf_path),
                "file_name": pdf_path.name,
                "element_type": "Text",
                "page_number": page_num + 1,
            }
        ))

        # Table extraction
        try:
            with pdfplumber.open(pdf_path) as pdf:
                plumber_page = pdf.pages[page_num]
                tables = plumber_page.extract_tables()
                for idx, table in enumerate(tables):
                    if table and len(table) > 1:
                        table_text = "\n".join(
                            [" | ".join(str(cell) if cell is not None else "" for cell in row)
                             for row in table]
                        )
                        all_documents.append(Document(
                            page_content=table_text,
                            metadata={
                                "source": str(pdf_path),
                                "file_name": pdf_path.name,
                                "element_type": "Table",
                                "page_number": page_num + 1,
                                "table_index": idx,
                            }
                        ))
        except Exception:
            continue

    doc.close()

print(f"\n✅ Extraction complete!")
print(f"Total Documents : {len(all_documents)}")
print(f"Text Blocks     : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Text')}")
print(f"Tables          : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Table')}")


Found 5 PDF files

Processing: Oracle.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Processing: Meta.pdf
Processing: Tesla.pdf
Processing: Nvidia.pdf
Processing: Apple.pdf

✅ Extraction complete!
Total Documents : 1043
Text Blocks     : 755
Tables          : 288


In [4]:
import yaml

def to_okf_concept(doc):
    company_guess = doc.metadata.get("file_name", "Unknown").split(".")[0].replace("_", " ")
    frontmatter = {
        "type": "FinancialTable" if doc.metadata.get("element_type") == "Table" else "FinancialText",
        "company": company_guess,
        "source_file": doc.metadata.get("file_name", "Unknown"),
        "page": doc.metadata.get("page_number", "?"),
    }
    fm_str = yaml.dump(frontmatter, sort_keys=False)
    doc.page_content = f"---\n{fm_str}---\n\n{doc.page_content.strip()}"
    return doc

all_documents = [to_okf_concept(d) for d in all_documents]
print(f"✅ Wrapped {len(all_documents)} documents as OKF concept blocks (frontmatter + content)")


✅ Wrapped 1043 documents as OKF concept blocks (frontmatter + content)


In [5]:
import subprocess
import sys

# 1. Wipe ALL related cached modules
to_remove = [k for k in sys.modules if any(x in k for x in 
    ["sentence", "langchain_huggingface", "huggingface", "langchain_core", "langchain"])]
for mod in to_remove:
    del sys.modules[mod]

# 2. Reinstall both together
subprocess.run(["pip", "install", "-q", "--no-cache-dir",
    "sentence-transformers==3.0.1",
    "langchain-huggingface==1.2.2"], check=True)

# 3. Verify sentence_transformers loads directly first
import sentence_transformers
print("sentence_transformers version:", sentence_transformers.__version__)

# 4. Now load HuggingFaceEmbeddings fresh
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
print("✅ Embedding model loaded!")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-08-10 06:44:14.783509: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786344255.237211     188 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786344255.347162     188 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786344256.412441     188 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

sentence_transformers version: 3.0.1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [6]:
pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 3.7 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
# CELL 6: Faster Semantic Chunking (GPU Optimized)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print(f"Current GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Use GPU with small batch size
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 2          # Small batch = less memory
    }
)

table_docs = [doc for doc in all_documents if doc.metadata.get("element_type") == "Table"]
text_docs = [doc for doc in all_documents if doc.metadata.get("element_type") == "Text"]

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,    
)

print("🔄 Performing semantic chunking on GPU...")
text_chunks = semantic_splitter.split_documents(text_docs)

chunks = table_docs + text_chunks

# Cleanup
del embeddings, semantic_splitter
torch.cuda.empty_cache()
gc.collect()

print(f"✅ Done!")
print(f"Tables: {len(table_docs)} | Text Chunks: {len(text_chunks)} | Total: {len(chunks)}")
print(f"GPU memory now: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

/tmp/ipykernel_188/160383551.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Current GPU memory: 0.41 GB


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 Performing semantic chunking on GPU...
✅ Done!
Tables: 288 | Text Chunks: 1848 | Total: 2136
GPU memory now: 0.42 GB


In [8]:
sample_embedding = embedding_model.embed_query(
    chunks[0].page_content
)

print("Embedding dimension:", len(sample_embedding))

Embedding dimension: 768


In [9]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "./financial_db" 
)

print("✅ Vector Store Created and Persisted")

✅ Vector Store Created and Persisted


In [10]:
query = "What is NVIDIA's total revenue?"
results = vectorstore.similarity_search(query,k = 3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata['file_name']} | Page: {doc.metadata.get('page_number')}")
    print(f"Type: {doc.metadata['element_type']}")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)


--- Result 1 ---
Source: Nvidia.pdf | Page: 94
Type: Text
---
type: FinancialText
company: Nvidia
source_file: Nvidia.pdf
page: 94
---

Table of Contents
NVIDIA Corporation and Subsidiaries
Notes to the Consolidated Financial Statements
(Continued)
We recognized revenue of $974 million and $729 million in fiscal years 2026 and 2025, respectively, that were included in the prior year
end deferred revenue balance. As of January 25, 2026, revenue related to remaining performance obligations from contracts greater than one year in length was $2.3
billion, ...

--- Result 2 ---
Source: Nvidia.pdf | Page: 57
Type: Table
---
type: FinancialTable
company: Nvidia
source_file: Nvidia.pdf
page: 57
---

Revenue | 100.0 | % |  | 100.0 | %
Cost of revenue | 28.9 |  |  | 25.0 | 
Gross profit | 71.1 |  |  | 75.0 | 
Operating expenses |  |  |  |  | 
Research and development | 8.6 |  |  | 9.9 | 
Sales, general and administrative | 2.1 |  |  | 2.7 | 
Total operating expenses | 10.7 |  |  | 12.6 | 
Opera

In [11]:
!pip install -q bitsandbytes accelerate

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1700,
    temperature=0.4,        
    top_p=0.90,
    do_sample=True,
    repetition_penalty=1.15,
    return_full_text=False,
)

# Plain LLM (used inside financial_rag() for the two-pass generation)
llm = HuggingFacePipeline(pipeline=pipe)

# Chat-formatted wrapper (used by the tool-calling agent below)
chat_llm = ChatHuggingFace(llm=llm)

print("mistralai/Mistral-7B-Instruct-v0.3 (4-bit).")
print("`llm`      -> plain text-completion, used by financial_rag()")
print("`chat_llm` -> chat + tool-calling wrapper, used by the agent")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0


mistralai/Mistral-7B-Instruct-v0.3 (4-bit).
`llm`      -> plain text-completion, used by financial_rag()
`chat_llm` -> chat + tool-calling wrapper, used by the agent


In [13]:
!pip install -q langchain langchain-community

In [14]:
import os
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ── ChatMessageHistory: moved to langchain-core in 1.x 
from langchain_core.chat_history import InMemoryChatMessageHistory
from types import SimpleNamespace

# Cross Encoder for reranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Per-session memory store
_memory_store = {}

def get_memory(company=None, session_id="default"):
    key = (company.lower().strip() if company else None, session_id)
    if key not in _memory_store:
        _memory_store[key] = InMemoryChatMessageHistory()
    return _memory_store[key]

def _strip_sec_header(text: str) -> str:
    if "[SEC FILING DATA]" not in text:
        return text.strip()
    parts = text.split("---\n", maxsplit=2)
    return parts[2].strip() if len(parts) >= 3 else text.strip()

def _clean_text(text):
    markers = ["<think>", "</think>", "**Final Answer**", "Final Answer:", "Changes made:"]
    for m in markers:
        if m in text:
            text = text.split(m)[0]
    return re.sub(r'\n+(I have|Note that|Please note).*', '', text,
                  flags=re.IGNORECASE | re.DOTALL).strip()

print("✅ Core utilities loaded")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Core utilities loaded


In [15]:
# Hybrid Retrieval (Semantic + BM25)
def hybrid_retrieval(query, vectorstore, company=None, k=50):
    # 1. Semantic search
    results = vectorstore.similarity_search_with_score(query, k=k * 3 if company else k)

    semantic_list = []
    for doc, score in results:
        if company and company.lower() not in doc.metadata.get("source", "").lower():
            continue
        semantic_list.append((doc, 1.0 / (1.0 + score)))

    # 2. BM25 keyword search
    all_data = vectorstore._collection.get(include=["documents", "metadatas"])
    filtered_texts, filtered_metas = [], []

    for text, meta in zip(all_data["documents"], all_data["metadatas"]):
        if company and company.lower() not in str(meta.get("source", "")).lower():
            continue
        filtered_texts.append(_strip_sec_header(text))
        filtered_metas.append(meta)

    bm25_list = []
    if filtered_texts:
        bm25 = BM25Okapi([t.lower().split() for t in filtered_texts])
        scores = bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:k]
        for i in top_idx:
            if scores[i] > 0:
                score_norm = scores[i] / max(scores.max(), 1)
                doc = SimpleNamespace(page_content=filtered_texts[i], metadata=filtered_metas[i])
                bm25_list.append((doc, score_norm))

    # Merge semantic + BM25
    merged = {id(d[0]): d for d in semantic_list}
    for doc, score in bm25_list:
        merged[id(doc)] = (doc, merged.get(id(doc), (None, 0))[1] + score * 0.7)

    return sorted(merged.values(), key=lambda x: x[1], reverse=True)[:k]


In [16]:
#  Reranking, multimodal boost, corrective RAG
def rerank_with_cross_encoder(query, candidates, top_n=15):
    if not candidates:
        return []
    docs  = [pair[0] for pair in candidates]
    texts = [_strip_sec_header(d.page_content) for d in docs]
    scores = cross_encoder.predict([[query, t] for t in texts])
    return sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_n]


def multimodal_boost(reranked_pairs):
    boosted = []
    for doc, score in reranked_pairs:
        new_score = float(score)
        et = doc.metadata.get("element_type", "")
        text = doc.page_content.lower()

        # Strong boost for real tables
        if et == "Table":
            new_score += 0.45

        # Extra boost for income-statement / segment pages
        keywords = [
            "year ended", "consolidated statements of income",
            "revenue by", "$ in millions", "gross profit",
            "operating income","operating expense", "net income"
        ]
        
        if any(k in text for k in keywords):
            new_score += 0.25

        boosted.append((doc, new_score))
    return sorted(boosted, key=lambda x: x[1], reverse=True)


def evaluate_retrieval_quality(query, docs):
    if not docs or len(docs) < 3:
        return False
    combined_text = " ".join(
        _strip_sec_header(doc.page_content)[:1000] for doc, _ in docs[:4]
    ).lower()
    query_words = [w for w in query.lower().split() if len(w) > 3]
    if not query_words:
        return True
    overlap = sum(1 for w in query_words if w in combined_text)
    required_overlap = max(2, len(query_words) // 3)
    print(f"   Retrieval Quality: {overlap}/{required_overlap} words matched")
    return overlap >= required_overlap

print("✅ Reranking + CRAG utilities loaded")


✅ Reranking + CRAG utilities loaded


In [17]:
# Conversation memory helpers
def _build_history_text(memory, max_turns=3):
    msgs  = memory.messages
    pairs = []
    i = 0
    while i < len(msgs) - 1:
        if msgs[i].type == "human" and msgs[i + 1].type == "ai":
            pairs.append((msgs[i].content, msgs[i + 1].content))
            i += 2
        else:
            i += 1
    recent = pairs[-max_turns:]
    if not recent:
        return ""
    lines = ["Previous conversation:"]
    for turn_idx, (q, a) in enumerate(reversed(recent), 1):
        short_a = a[:500] + "…" if len(a) > 500 else a
        lines.append(f"\n[Turn {turn_idx}] User: {q}")
        lines.append(f"AI: {short_a}")
    return "\n".join(lines)

print("✅ Memory helpers loaded")


✅ Memory helpers loaded


In [18]:
# Main financial_rag() function 
def financial_rag(query: str, company: str = None, session_id: str = "default"):
    global vectorstore, embedding_model, llm

    if not all([vectorstore, embedding_model, llm]):
        return "❌ Error: vectorstore, embedding_model or llm not initialized."

    memory = get_memory(company, session_id)

    # Truncate query for clean logging (no prompt leak)
    query_preview = (query.strip()[:60] + "...") if len(query.strip()) > 60 else query.strip()
    print(f"🔍 Company: {company or 'All'} | Query: {query_preview}")

    # ── Retrieval ──────────────────────────────────────────────────────────
    candidates = hybrid_retrieval(query, vectorstore, company=company, k=40)
    reranked   = rerank_with_cross_encoder(query, candidates, top_n=7)
    reranked   = multimodal_boost(reranked)

    # ── Corrective RAG ─────────────────────────────────────────────────────
    if not evaluate_retrieval_quality(query, reranked):
        print("⚠️  Corrective RAG triggered — widening search...")
        candidates = hybrid_retrieval(query, vectorstore, company=company, k=50)
        reranked   = rerank_with_cross_encoder(query, candidates, top_n=8)
        reranked   = multimodal_boost(reranked)

    # ── Filter & cap ───────────────────────────────────────────────────────
    filtered_docs = [
        (doc, score) for doc, score in reranked
        if not company or company.lower() in str(doc.metadata.get("source", "")).lower()
    ][:16]

    if len(filtered_docs) < 3:
        return f"❌ Not enough relevant information found for '{company}'."

    context = "\n\n---\n\n".join(_strip_sec_header(doc.page_content) for doc, _ in filtered_docs)

    # ── Pass 1: Generate Response ───────────────────────────────────────────
    print("📝 Pass 1: Generating Response")
    pass1_prompt = f"""You are a Senior Institutional Financial Analyst.

ABSOLUTE RULES — NEVER BREAK THEM:

1. Use ONLY numbers that appear VERBATIM in the Context below.
2. Use the context provide below
3. Never invent, estimate, round, or pull any number from memory or training data.
4. LABEL LOCKING (critical):
   - Every number must stay attached to the exact same label it has in the Context.
   - Operating Expenses numbers can ONLY be used for Operating Expenses.
   - Operating Income numbers can ONLY be used for Operating Income.
   - Revenue numbers can ONLY be used for Revenue.
   - Gross Profit / Gross Margin numbers can ONLY be used for Gross Profit / Gross Margin.
   - Net Income numbers can ONLY be used for Net Income.
   - Research & Development, SG&A, and other line items must also keep their own numbers.
   - Never swap or mix numbers between different metrics.

5. When you write a number, always pair it with its full correct label, for example:
   “Operating expenses were $23,076 million”
   “Operating income was $130,387 million”
   Never write a bare number and later assign it to a different metric.

6. DIRECTION RULE:
   - Before writing “increased”, “decreased”, “rose”, “declined”, “growth”, or “drop”, 
     first compare the two numbers belonging to the SAME metric.
   - Later > Earlier → must say “increased” or “rose”.
   - Later < Earlier → must say “decreased” or “declined”.
   - Never assume direction from the movement of expenses or from surrounding text.

7. If either year is missing for a metric, write exactly: “percentage change not available in the retrieved sections.”

9. If a metric is not present in the Context, write: “not disclosed in the retrieved sections.”

11. Write in clear Financial tone

FIRST, decide the type of question:

A. If the question is mainly about financial figures, trends, revenue, gross margins and profits, expenses, income, cash flow, etc.:
   → Present the key figures in a clean Markdown table with columns such as:
     | Metric                  | Earlier Year | Later Year | Change | % Change          |
     |-------------------------|--------------|------------|--------|-------------------|
   → After the table, write a structured analysis using only the numbers from the table.
   → For every % Change, show the calculation (example: ((130387-81453)/81453 × 100 = 60.1%)).
     
B. If the question is about architecture, technology, products (Blackwell, Rubin, etc.), strategy, risks, competition,competators, outlook, or any non-numeric topic:
   → Do NOT create a financial table.
   → Directly write a clear, structured analysis based on the Context.
   → Only mention numbers if they are relevant and present in the Context.

**Context:**
{context}

**Question:**
{query}

**Output Format:**
Provide a structured, concise, and accurate financial analysis.

Financial Analysis:"""

    raw_pass1 = llm.invoke(pass1_prompt)
    final_response = raw_pass1.content if hasattr(raw_pass1, "content") else str(raw_pass1)
    final_response = _clean_text(final_response)


   
    # ── Memory ─────────────────────────────────────────────────────────────
    memory.add_user_message(query)
    memory.add_ai_message(final_response)

    # ── Sources ────────────────────────────────────────────────────────────
    sources = [
        f"{doc.metadata.get('file_name', 'Unknown')} | Page {doc.metadata.get('page_number', '?')}"
        for doc, _ in filtered_docs
    ]
    unique_sources = list(dict.fromkeys(sources))

    final_output = (
        f"# Financial Analysis — {company or 'All Companies'}\n\n"
        + final_response
        + "\n\n## Sources\n"
        + "\n".join(f"- {s}" for s in unique_sources)
    )

    # Debug metadata
    financial_rag._last_context = context
    financial_rag._last_sources = unique_sources
    financial_rag._debug = {
        "initial_docs": len(candidates),
        "final_docs": len(filtered_docs),
        "corrective_triggered": not evaluate_retrieval_quality(query, reranked),
    }

    return final_output

print("✅ financial_rag() with refinement pass ready")


✅ financial_rag() with refinement pass ready


In [19]:
# CELL 17b: BERTScore — measures how well the generated answer is grounded in retrieved context 
from bert_score import score as bert_score

_bert_log = []  # stores {query, company, precision, recall, f1} for every call

def compute_bertscore(reference_text: str, candidate_text: str):
    """
    BERTScore of candidate_text against reference_text.
    Here reference = retrieved source context, candidate = generated answer.

    Unlike BLEU, this compares contextual embeddings of tokens instead of
    exact word matches, so a well-paraphrased but accurate answer still
    scores high. Used here as a grounding/faithfulness proxy:
      - High F1  -> answer's meaning is well supported by the retrieved context
      - Low F1   -> answer may be drifting from / hallucinating beyond the context

    Returns (precision, recall, f1) as plain floats.
    """
    if not reference_text.strip() or not candidate_text.strip():
        return 0.0, 0.0, 0.0

    # BERTScore compares sentence-by-sentence internally; long inputs are fine,
    # but we truncate extremely long context purely to keep this fast.
    ref = reference_text[:4000]
    cand = candidate_text[:4000]

    P, R, F1 = bert_score(
        [cand], [ref],
        lang="en",
        model_type="distilbert-base-uncased",
        verbose=False,
    )
    return P.item(), R.item(), F1.item()


def financial_rag_with_bertscore(query: str, company: str = None, session_id: str = "default"):
    """
    Thin wrapper around financial_rag() that additionally computes and prints
    a BERTScore for the generated response (answer vs. retrieved context),
    and logs it to _bert_log for later inspection / averaging.
    """
    response = financial_rag(query, company=company, session_id=session_id)

    context = getattr(financial_rag, "_last_context", "")
    precision, recall, f1 = compute_bertscore(context, response) if context else (0.0, 0.0, 0.0)

    print(f"📊 BERTScore (answer vs. retrieved context) — Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    _bert_log.append({
        "query": query.strip()[:80],
        "company": company or "All",
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    })

    return response


# Backward-compatible alias, in case earlier cells still call the old name
financial_rag_with_bleu = financial_rag_with_bertscore


def show_bertscore_log():
    """Pretty-print the BERTScore for every query run so far."""
    if not _bert_log:
        print("No queries logged yet.")
        return
    print(f"{'#':<3} {'Company':<10} {'Precision':<10} {'Recall':<10} {'F1':<8} Query")
    print("-" * 100)
    for i, entry in enumerate(_bert_log, 1):
        print(f"{i:<3} {entry['company']:<10} {entry['precision']:<10} {entry['recall']:<10} {entry['f1']:<8} {entry['query']}")
    avg_p = sum(e['precision'] for e in _bert_log) / len(_bert_log)
    avg_r = sum(e['recall'] for e in _bert_log) / len(_bert_log)
    avg_f1 = sum(e['f1'] for e in _bert_log) / len(_bert_log)
    print("-" * 100)
    print(f"Average -> Precision: {avg_p:.4f} | Recall: {avg_r:.4f} | F1: {avg_f1:.4f}")

print("✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores")
print("✅ Call show_bertscore_log() anytime to see all scores so far")


✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores
✅ Call show_bertscore_log() anytime to see all scores so far


In [20]:
from IPython.display import display, Markdown

query = """Provide a comprehensive financial analysis of NVIDIA using the latest SEC 10-K filing.

Focus on:
- Total revenue breakdown and year-over-year growth (Data Center vs Gaming vs Professional Visualization vs Automotive)
- Data Center segment performance, including AI infrastructure demand drivers
- Discuss What are the Gross margin trends and key factors affecting profitability
- Discuss what are the Operating expenses, operating income, and net income trends
- Cash flow generation, capital expenditures, and liquidity position
- Key financial highlights and management commentary on future outlook


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""



response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Provide a comprehensive financial analysis of NVIDIA using t...
   Retrieval Quality: 20/25 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 20/25 words matched


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

📊 BERTScore (answer vs. retrieved context) — Precision: 0.7583 | Recall: 0.7785 | F1: 0.7683


# Financial Analysis — Nvidia

NVIDIA Corporation (NVDA)
=============================================

### Total Revenue Breakdown and Growth

In the latest SEC 10-K filing, NVIDIA reported total revenue of $215,938 million for the year ended January 25, 2026, representing a substantial increase of 65.4% compared to $130,497 million in the previous year. This growth can be attributed to various factors, including strong demand for data center computing solutions and the expansion of NVIDIA’s product portfolio beyond traditional PC graphics.

The following table illustrates the revenue distribution across different end markets for the years 2024, 2025, and 2026:

| Metric                | 2024 | 2025 | 2026 | Change | % Change |
|-----------------------|------|------|------|--------|----------|
| Data Center           | $47,525 | $115,186 | $193,737 | $148,552 | 133.4% |
| Compute              | $38,950 | $102,196 | $162,361 | $59,165 | 57.3% |
| Networking            | $8,575 | $12,990 | $31,376 | $22,781 | 181.9% |
| Gaming               | $10,447 | $11,350 | $16,042 | $5,595 | 52.4% |
| Professional Visualization | $1,553 | $1,878 | $3,191 | $1,638 | 85.5% |
| Automotive             | $1,091 | $1,694 | $2,349 | $1,258 | 74.2% |
| OEM and Other         | $306 | $389 | $619 | $313 | 80.7% |
| **Total Revenue**    | $60,922 | $130,497 | $215,938 | $85,441 | 65.4% |

It is evident that the Data Center segment dominates NVIDIA’s revenue, accounting for more than half of the overall revenue ($193,737 million out of $215,938 million). The significant growth in the Data Center segment (133.4%) is primarily driven by the increasing adoption of NVIDIA’s Blackwell computing platform for accelerated computing and AI solutions.

### Data Center Segment Performance

The Data Center segment represents NVIDIA’s strategic focus on becoming a data center-scale AI infrastructure provider. According to the company, the growth in the Data Center segment was fueled by the major platform shifts towards accelerated computing and AI. Specifically, the availability of data centers, energy, and capital to support the buildout of NVIDIA AI infrastructure by customers and partners is crucial for continued success. Any potential shortages of these resources could negatively impact future revenue and financial performance.

Expanding energy capacity to meet demand is a complex, multi-year process involving significant regulatory, technical, and construction challenges. Access to capital can be particularly limited for less-capitalized companies, potentially causing difficulties in securing financing for large-scale infrastructure projects.

### Gross Margin Trends and Key Factors Affecting Profitability

Gross profit increased from $44,301 million in 2024 to $153,463 million in 2026, marking a notable improvement of 258.5%. This growth can be attributed to the expanding product offerings and increased sales volumes, especially within the Data Center segment.

However, the gross margin percentage decreased slightly from 71.1% in 2025 to 71.0% in 2026. This decrease can be explained by the shift towards lower-margin Data Center products, which tend to have thinner gross margins compared to gaming and professional visualization products. Despite this marginal decline, NVIDIA still maintains one of the highest gross margins among semiconductor companies.

### Operating Expenses, Operating Income, and Net Income Trends

Operating expenses increased from $11,329 million in 2024 to $23,076 million in 2026, representing a 101.4% rise. This growth is primarily due to increased research and development (R&D) spending, as well as higher sales, general, and administrative (SG&A) costs associated with supporting the growing business.

Despite the rise in operating expenses, operating income improved substantially, rising from $32,972 million in 2024 to $130,387 million in 2026, demonstrating a 286.3% increase. This impressive growth indicates that NVIDIA continues to effectively manage its expenses while benefiting from robust revenue growth.

Net income also saw a considerable jump, increasing from $29,760 million in 2024 to $120,067 million in 2026, corresponding to a 306.8% increase. This growth is indicative of the company's ability to convert top-line revenue growth into bottom-line earnings.

### Cash Flow Generation, Capital Expenditures, and Liquidity Position

NVIDIA reported net cash provided by operating activities of $102,718 million in 2026, up from $64,089 million in 2025. This improvement is largely due to the strong revenue growth experienced during the period.

Capital expenditures rose from $20,421 million in 2025 to $52,228 million in 2026, primarily due to investments in manufacturing facilities, equipment, and research and development initiatives.

As of January 25, 2026, NVIDIA held $62.6 billion in cash, cash equivalents, and marketable securities, providing ample liquidity to meet operational needs for at least the next twelve months and beyond.

### Key Financial Highlights and Management Commentary on Future Outlook

Some key financial highlights from the latest SEC 10-K filing include:

* Total revenue of $215,938 million, up 65.4% YoY
* Data Center segment revenue of $193,737 million, up 133.4% YoY
* Gross profit of $153,463 million, up 258.5% YoY
* Operating income of $130,387 million, up 286.3% YoY
* Net income of $120,067 million, up 306.8% YoY

Management expects continued growth in the Data Center segment, driven by ongoing demand for AI infrastructure and the expansion of NVIDIA's product portfolio. However, potential challenges include regulatory hurdles, energy capacity limitations, and capital market volatility. Investors should closely monitor these factors when considering

## Sources
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 106
- Nvidia.pdf | Page 72
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 60
- Nvidia.pdf | Page 53

In [21]:
print(financial_rag._last_context)

---
type: FinancialText
company: Nvidia
source_file: Nvidia.pdf
page: 71
---

Table of Contents
NVIDIA Corporation and Subsidiaries
Consolidated Statements of Income
(In millions, except per share data)
Year Ended
Jan 25, 2026
Jan 26, 2025
Jan 28, 2024
Revenue
$
215,938 
$
130,497 
$
60,922 
Cost of revenue
62,475 
32,639 
16,621 
Gross profit
153,463 
97,858 
44,301 
Operating expenses
Research and development
18,497 
12,914 
8,675 
Sales, general and administrative
4,579 
3,491 
2,654 
Total operating expenses
23,076 
16,405 
11,329 
Operating income
130,387 
81,453 
32,972 
Interest income
2,300 
1,786 
866 
Interest expense
(259)
(247)
(257)
Other income, net
9,022 
1,034 
237 
Total other income, net
11,063 
2,573 
846 
Income before income tax
141,450 
84,026 
33,818 
Income tax expense
21,383 
11,146 
4,058 
Net income
$
120,067 
$
72,880 
$
29,760 
Net income per share:
Basic
$
4.93 
$
2.97 
$
1.21 
Diluted
$
4.90 
$
2.94 
$
1.19 
Weighted average shares used in per share compu

In [22]:
query = "Discuss about the architecture of Blackwell and Rubin and how they are benefits to Nvidia in depth "

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Discuss about the architecture of Blackwell and Rubin and ho...
   Retrieval Quality: 1/3 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 1/3 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.8352 | Recall: 0.8321 | F1: 0.8336


# Financial Analysis — Nvidia

The NVIDIA Blackwell and Rubin architectures represent significant advancements in the company's data center-scale computing platform. Both architectures incorporate a combination of hardware components, such as GPUs, CPUs, DPUs, interconnects, switch chips, and systems, as well as software stacks and algorithms.

The NVIDIA Blackwell architecture, launched in fiscal year 2025, offers exceptional performance and efficiency for cutting-edge generative AI and accelerated computing workloads. It serves various industries and use cases due to its adaptability. Notably, it connects 36 Grace CPUs and 72 Blackwell GPUs in a data center scale, liquid-cooled design, enabling real-time trillion-parameter inference and training.

In fiscal year 2026, NVIDIA introduced the NVIDIA Blackwell Ultra platform, optimized for agentic, reasoning, and physical AI. Building upon the architectural breakthroughs of Blackwell and leveraging Dynamo inference software, it delivers a substantial increase in token throughput and reduction in cost per token compared to the Hopper generation.

Introduced in fiscal year 2026, the NVIDIA Rubin platform is expected to commence production shipments in the second half of fiscal year 2027. Designed for agentic AI and reasoning, it excels at processing multi-step problem-solving and massive long-context workflows, offering a 10x reduction in cost per token compared to Blackwell.

By developing these innovative architectures, NVIDIA aims to maintain its competitive advantage in the data center computing market. The Blackwell and Rubin architectures demonstrate NVIDIA's commitment to continuous innovation, pushing the boundaries of performance and efficiency in AI and accelerated computing workloads.

| Metric                  | Fiscal Year 2025 (Earlier)       | Fiscal Year 2026 (Later)        | Change               | % Change           |
|-------------------------|----------------------------------|-----------------------------|----------------------|---------------------|
| Token Throughput       | Not Disclosed                    | Blackwell Ultra              | Not Available         | Not Available       |
| Cost Per Token         | Not Disclosed                    | Blackwell -> Blackwell Ultra | Decrease             | 10%                 |
| Cost Per Token         | Blackwell                         | Rubin                       | Decrease             | 10%                 |

## Sources
- Nvidia.pdf | Page 7
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 9

In [23]:

# CELL 26: NVIDIA — Export Control Risks

from IPython.display import display, Markdown

query = """Provide a detailed analysis of NVIDIA's export control risks, geopolitical exposure, and China-related challenges.

Focus on:
- Impact of U.S. export restrictions on products
- Licensing requirements, revenue impact, and inventory charges
- Competitive effects on China data center market discuss this in detail
- Mitigation strategies and long-term implications
- Broader supply chain and regulatory risks discuss this in detail

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Quote relevant sections from the Risk Factors and Business sections."""

# Run the analysis
response = financial_rag_with_bleu(query, company="Nvidia")

# Display the response
display(Markdown(response))


🔍 Company: Nvidia | Query: Provide a detailed analysis of NVIDIA's export control risks...
   Retrieval Quality: 20/23 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 20/23 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.8091 | Recall: 0.8166 | F1: 0.8128


# Financial Analysis — Nvidia

NVIDIA's Export Control Risks, Geopolitical Exposure, and China-Related Challenges
===============================================================================================

NVIDIA faces various risks related to export controls, geopolitics, and China-specific challenges that impact its business and financial results. This section provides a comprehensive analysis of these factors, focusing on the impact of U.S. export restrictions on products, licensing requirements, revenue impacts, and competitive effects on the China data center market. Additionally, we discuss mitigation strategies and long-term implications, as well as broader supply chain and regulatory risks.

Impact of U.S. Export Restrictions on Products
---------------------------------------------

Export controls targeting GPUs and semiconductors associated with AI have subjected and may in the future subject downstream users of NVIDIA's products to restrictions on the use, resale, repair, or transfer of its products, negatively impacting the company's business and financial results ("Risk Factors" Section). These controls may disrupt the supply and distribution chain for a substantial portion of NVIDIA's products, which are warehoused in and distributed from Hong Kong ("Business" Section). Export controls restricting NVIDIA's ability to sell data center GPUs may also negatively impact demand for its networking products used in servers containing its GPUs. The USG may also impose export controls on NVIDIA's networking products, such as high-speed network interconnects, to limit the ability of downstream parties to create large clusters for frontier model training.

Licensing Requirements, Revenue Impact, and Inventory Charges
-----------------------------------------------------------

If NVIDIA is able to sell licensed products into the China market, it may not be able to pass along all or any of the tariff to its customers, and may be subject to litigation, increased costs, and a harmed competitive position ("Risk Factors" Section). The licensing process may not be resolved before significant business opportunities evaporate, even if the USG grants any requested licenses. The licenses have already and may in the future be temporary, impose burdensome conditions regarding the installation, maintenance, and use of such products, or include financial or economic requirements that NVIDIA or its customers or end users cannot or choose not to fulfill. The licensing requirements have already and may in the future benefit certain of NVIDIA's competitors, as the licensing process will make its pre-sale and post-sale technical support efforts more cumbersome and less certain and encourage customers in China, the Middle East, and other regions to pursue alternatives to NVIDIA's products, including semiconductor suppliers based in China, Europe, and Israel.

Competitive Effects on China Data Center Market
----------------------------------------------

As of the end of fiscal year 2026, NVIDIA was effectively foreclosed from competing in China's data center computing/compute market, and its effective foreclosure from the China market helped its competitors build larger developer and customer ecosystems to challenge NVIDIA worldwide. Unless NVIDIA is able to return with a product that meets the approval of both the USG and the Chinese government, its lost opportunity and the benefit to its competitors will have a material and adverse impact on its business, operating results, and financial condition ("Risk Factors" Section).

Mitigation Strategies and Long-Term Implications
----------------------------------------------

To mitigate these risks, NVIDIA is working to enhance the resiliency and redundancy of its supply chain, which is currently concentrated in Asia. However, new and existing export controls or changes to existing export controls could limit alternative manufacturing locations and negatively impact the company's business. NVIDIA is also addressing regulatory risks through engagement with global policymakers, industry associations, and standards organizations to advocate for policies that promote innovation and growth while ensuring appropriate safeguards.

Broader Supply Chain and Regulatory Risks
------------------------------------------

Compliance with laws, rules, and regulations has not otherwise had a material effect upon NVIDIA's capital expenditures, results of operations, or competitive position, but could further increase costs, impact its competitive position, and otherwise may have a material adverse impact on its business, financial condition, and results of operations in subsequent periods ("Risk Factors" Section). Regulators in China have inquired about NVIDIA's sales and efforts to supply the China market and its fulfillment of the commitments it entered at the close of its Mellanox acquisition. If regulators conclude that NVIDIA has failed to fulfill the terms of its Mellanox acquisition or violated any applicable law in China, it could be subject to financial penalties, restrictions on its ability to conduct its business, restrictions or other orders regarding its networking business, products, and services, or otherwise impact its operations in China, any of which could have a material and adverse impact on its business, operating results, and financial condition.

Advice for Investors
--------------------

Investors should consider the potential impact of export controls, geopolitical risks, and China-related challenges on NVIDIA's business and financial results when making investment decisions. They should monitor developments in these areas closely and assess how they might affect the company's short-term and long-term prospects. Additionally, investors should evaluate NVIDIA's strategies for navigating these risks and its ability to adapt to changing regulatory environments.

## Sources
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 15
- Nvidia.pdf | Page 41
- Nvidia.pdf | Page 37

In [24]:
query = """Evaluate NVIDIA's  positioning across its markets.

Focus on:
- Competition in Data Center (AMD, Intel, custom ASICs from hyperscalers)
- Discuss about Gaming GPU competition 
- Professional Visualization and Automotive segments
- Overall technology leadership in GPUs, CUDA, networking, and software
- Barriers to entry and ecosystem strength

Discuss strengths, weaknesses, and investor implications with references from the filing."""

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's  positioning across its markets.

Focus on...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 18/14 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 18/14 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7424 | Recall: 0.7724 | F1: 0.7571


# Financial Analysis — Nvidia

| Metric                  | Earlier Year | Later Year | Change | % Change          |
|-------------------------|--------------|------------|--------|-------------------|
| **Data Center Market**  |              |            |        |                   |
| Revenue                | Not Disclosed | Not Disclosed | Not Available | Not Available       |
| Gross Profit           | Not Disclosed | Not Disclosed | Not Available | Not Available       |
| **Gaming Market**      |              |            |        |                   |
| Revenue                | Not Disclosed | Not Disclosed | Not Available | Not Available       |
| Gross Profit           | Not Disclosed | Not Disclosed | Not Available | Not Available       |
| **Professional Visualization** |              |            |        |                   |
| Revenue                | Not Disclosed | Not Disclosed | Not Available | Not Available       |
| Gross Profit           | Not Disclosed | Not Disclosed | Not Available | Not Available       |
| **Automotive Segment** |              |            |        |                   |
| Revenue                | Not Disclosed | Not Disclosed | Not Available | Not Available       |
| Gross Profit           | Not Disclosed | Not Disclosed | Not Available | Not Available       |

**Analysis:**

In the Data Center market, NVIDIA faces competition from established players like AMD and Intel, as well as potential threats from custom ASICs developed by hyperscalers. However, NVIDIA's unique selling point lies in its full-stack innovation approach, addressing various end markets with a unified underlying architecture leveraging GPUs, CPUs, CUDA, and networking technologies. This strategy allows NVIDIA to deliver order-of-magnitude performance advantages compared to legacy approaches in targeted markets.

For the Gaming market, NVIDIA competes directly with AMD and Intel, as well as other manufacturers offering GPUs for gaming desktops and laptops. NVIDIA's GeForce RTX series GPUs, featuring ray tracing technology, deep learning super sampling (DLSS), and tensor core technology, give it a strong edge in delivering high-quality gaming experiences.

The Professional Visualization segment benefits from NVIDIA's close collaboration with Independent Software Vendors (ISVs). By optimizing their offerings for NVIDIA GPUs, NVIDIA enables productivity gains and introduces new capabilities for critical workflows in numerous industries. Furthermore, the increasing integration of AI in professional applications makes NVIDIA's RTX PRO GPUs even more valuable due to their Tensor Core technology found in Data Center solutions.

In the Automotive segment, NVIDIA aims to leverage its expertise in AI, GPUs, and networking to address opportunities in autonomous driving and other automotive applications. However, specific details about the competition and revenue generated in this segment are not disclosed in the provided filings.

Overall, NVIDIA maintains a strong technology leadership position in GPUs, CUDA, networking, and software. Its extensive ecosystem, including partnerships with numerous ISVs, gives it a competitive advantage by enabling a wide range of applications across multiple industries.

Barriers to entry for competing with NVIDIA in its key markets include the complexity of developing cutting-edge technology, the need for significant investment in research and development, and the necessity of building a robust ecosystem of partners and collaborations.

From an investor perspective, NVIDIA's strategic focus on delivering performance leaps beyond Moore's Law, combined with its innovative full-stack approach, positions it well for long-term success in its targeted markets. However, investors should remain aware of the intensifying competition, especially in the Data Center market, and the potential risks associated with regulatory scrutiny and export controls.

## Sources
- Nvidia.pdf | Page 9
- Nvidia.pdf | Page 12
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 7
- Nvidia.pdf | Page 5

In [25]:
query = """Analyze NVIDIA's long-term corporate strategy, key risks, and growth outlook.

Focus on:
- Platform strategy (hardware + software + ecosystem)
- Expansion into AI, robotics, autonomous driving, and professional visualization
- Supply chain, manufacturing, and capacity risks
- Human capital, R&D investment, and innovation approach
- Major risks from the Risk Factors section and mitigation efforts

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Provide a balanced view with exact quotes and key takeaways for long-term investors."""


response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Analyze NVIDIA's long-term corporate strategy, key risks, an...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 22/22 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 22/22 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7571 | Recall: 0.7633 | F1: 0.7602


# Financial Analysis — Nvidia

Long-Term Strategy, Risks, and Outlook for NVIDIA
=====================================================================

NVIDIA's long-term corporate strategy revolves around advancing its accelerated computing platform, which enables solving complex problems more efficiently compared to traditional computational methods. The platform strategy encompasses hardware, software, and an extensive ecosystem.

### Hardware + Software + Ecosystem Strategy

##### Hardware

* Leverages GPU architecture to create platforms for scientific computing, AI, data science, autonomous vehicles, robotics, and digital twin applications.
* Focus on Data Center, Gaming, Professional Visualization, and Autonomous Vehicles markets.
* Unique programmable nature of the architecture allows for shared underlying technology across multiple end markets via various software stacks.

##### Software

* Offers NVIDIA AI Enterprise—a comprehensive software suite designed to simplify the development and deployment of production-grade, end-to-end generative AI applications.
* Includes NVIDIA NIM, NVIDIA NeMo, and AI Blueprints to optimize, evaluate, and safeguard domain-adapted models.
* Enables organizations to securely develop and run AI applications on NVIDIA-accelerated infrastructure anywhere.

##### Ecosystem

* Large and expanding ecosystem supporting the AI technology leadership.
* Provides an open, modular DRIVE software platform for autonomous driving, mapping, and parking services, and intelligent in-vehicle experiences.

### Expansion into Key Areas

* **AI**: Complete, end-to-end accelerated computing platform for AI, addressing both training and inferencing.
* **Robotics**: Not explicitly mentioned in the provided context, but the platform strategy suggests potential expansion opportunities.
* **Autonomous Driving**: Running an in-vehicle operating system (DRIVE OS), a reference sensor set, and an open, modular DRIVE software platform.
* **Professional Visualization**: Not explicitly mentioned in the provided context, but the platform strategy suggests potential expansion opportunities.

### Supply Chain, Manufacturing, and Capacity Risks

* Access to capital can be particularly constrained for less-capitalized companies, which may face difficulties securing financing for large-scale infrastructure projects.
* Expanding energy capacity to meet demand is a complex, multi-year process involving significant regulatory, technical, and construction challenges.
* Increased compliance costs due to changes or increases in antitrust legislation, regulation, administrative rule making, and increased focus from regulators on cybersecurity vulnerabilities and risks.
* Regulatory scrutiny in various regions, including the EU, US, UK, South Korea, Japan, and China, regarding sales of GPUs and other NVIDIA products, allocation of supply, foundation models, investments, partnerships, strategies, roadmaps, and agreements with customers, suppliers, and partners.

### Human Capital, R&D Investment, and Innovation Approach

* Emphasizes full-stack innovation across architecture, chip design, system, interconnect, algorithm, and software layers to deliver order-of-magnitude performance advantages in target markets.
* Invests heavily in research and development in markets where there is limited operating history, which may not produce meaningful revenue for several years, if at all.

### Major Risks and Mitigation Efforts

* Shortage of energy or other necessary resources for data centers could impact future revenue and financial performance.
* Compliance costs may increase due to changes or increases in antitrust legislation, regulation, administrative rule making, and increased focus from regulators on cybersecurity vulnerabilities and risks.
* Regulatory scrutiny in various regions, including the EU, US, UK, South Korea, Japan, and China, regarding sales of GPUs and other NVIDIA products, allocation of supply, foundation models, investments, partnerships, strategies, roadmaps, and agreements with customers, suppliers, and partners.
* Mitigation efforts include complying with regulations, providing requested information when required, and continuing to invest in research and development.

For long-term investors, understanding NVIDIA's strategic focus on advanced computing platforms, expansion into AI, robotics, autonomous driving, and professional visualization, along with associated risks, provides valuable insights. It is crucial to monitor regulatory developments, supply chain issues, and human capital management to assess the company's long-term prospects.

Investors may consider diversifying their portfolios to manage risk exposure, focusing on companies with complementary strengths and strategies. Additionally, staying informed about advancements in AI, robotics, and autonomous driving sectors can help investors better understand NVIDIA's position within these markets. Regularly reviewing financial reports, earnings calls, and analyst opinions can also aid in making informed decisions.

## Sources
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 9

In [26]:
query = "explain the supply chain risk of Nvidia in more depth like who are the key suppliers and what they supply and why it is cruical for Nvidia and also tell based on context suggest improvements in depth."

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))


🔍 Company: Nvidia | Query: explain the supply chain risk of Nvidia in more depth like w...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 10/7 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 10/7 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7878 | Recall: 0.7678 | F1: 0.7777


# Financial Analysis — Nvidia

Supply Chain Risks for Nvidia:

Key Suppliers:
1. Samsung Electronics Co., Ltd. – Provides DRAM and Foundry Services.
   - Crucial for Nvidia's production of GPUs, as DRAM is essential for storing temporary data during computation.
   - Foundry services are crucial for manufacturing custom silicon designs, ensuring the quality and efficiency of Nvidia's chips.

2. TSMC (Taiwan Semiconductor Manufacturing Company) – Provides Foundry Services.
   - Similar to Samsung, TSMC provides foundry services for manufacturing Nvidia's chips.
   - Offers advanced process nodes, enabling Nvidia to produce smaller, more efficient, and powerful chips.

Disruption Potential:
The primary risks stemming from these key suppliers revolve around their ability to deliver on schedule, maintain quality, and comply with export regulations. Delays in delivery can lead to production halts, while poor quality can result in increased rework costs and lower product performance. Compliance issues with export regulations, particularly those imposed by the US Government, can restrict the sale and distribution of Nvidia's products, negatively impacting demand and financial results.

Improvement Strategies:
To mitigate these risks, Nvidia should consider diversifying its supplier base to minimize reliance on a single source. This can be achieved by partnering with alternative foundries, such as GlobalFoundries or Intel, to manufacture chips. Additionally, building stronger relationships with current suppliers can help negotiate better terms, improve communication, and collaborate on innovation to address potential challenges. Lastly, maintaining a robust inventory management system will enable Nvidia to respond to shortages caused by supplier disruptions without significantly impacting overall production.

## Sources
- Nvidia.pdf | Page 27
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 34
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 24

In [27]:
query = "Discuss about operating expenses(R&D and SG&A) of Nvidia in depth and also discuss about trends"

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))


🔍 Company: Nvidia | Query: Discuss about operating expenses(R&D and SG&A) of Nvidia in ...
   Retrieval Quality: 2/3 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 2/3 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7860 | Recall: 0.8021 | F1: 0.7940


# Financial Analysis — Nvidia

| Metric                  | Earlier Year | Later Year | Change | % Change          |
|-------------------------|--------------|------------|--------|-------------------|
| Research and development | $12,914       | $18,497    | $5,583 | 43.0%             |
| Sales, general and administrative | $3,491        | $4,579    | $1,088 | 31.0%            |
| Total operating expenses | $16,405      | $23,076   | $6,671 | 41.0%              |

From the given financial data, we observe the operating expenses of Nvidia for the years 2024, 2025, and 2026. We will focus on the Research and Development (R&D) and Sales, General and Administrative (SG&A) expenses.

For R&D expenses, there was an increase from $12,914 million in 2024 to $18,497 million in 2026, representing a growth of $5,583 million or 43.0%. This rise suggests that Nvidia invested more in research and development during these years.

Similarly, for SG&A expenses, there was an increase from $3,491 million in 2024 to $4,579 million in 2026, resulting in a growth of $1,088 million or 31.0%. This indicates that Nvidia also increased spending on general and administrative expenses during the analyzed period.

Trends:

* Both R&D and SG&A expenses have shown a consistent upward trend since 2024, indicating an increasing investment in innovation and operational efficiency by Nvidia.
* The percentage changes suggest that the rate of growth in both expenses has been higher for SG&A compared to R&D, which may indicate a shift towards focusing more on administration aspects. However, the absolute dollar amounts still favor R&D investments.

## Sources
- Nvidia.pdf | Page 105
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 58
- Nvidia.pdf | Page 104
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 75

In [28]:
query = """Evaluate NVIDIA's overall corporate strategy and capital allocation decisions.

Focus on:
- Core business model and diversification efforts
- R&D investment trends
- Mergers & acquisitions strategy
- Share buyback and dividend policy
- Long-term vision in AI, robotics, autonomous vehicles, and Omniverse
- Management's capital allocation priorities

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 

Provide investor implications."""

response = financial_rag_with_bleu(query, company="Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's overall corporate strategy and capital all...
   Retrieval Quality: 11/18 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 11/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7979 | Recall: 0.7751 | F1: 0.7863


# Financial Analysis — Nvidia

NVIDIA Corporation Strategy and Capital Allocation Decisions
===============================================================================

Overview
---------

NVIDIA Corporation, a leading technology company, focuses on accelerated computing to tackle complex computational challenges across various domains. Its core business revolves around providing hardware and software solutions for artificial intelligence (AI), data analytics, scientific computing, robotics, autonomous vehicles, and 3D graphics. The company's strategic approach encompasses a strong commitment to innovation, technological advancements, and strategic partnerships.

Core Business Model and Diversification Efforts
--------------------------------------------------

NVIDIA's initial focus was on PC graphics; however, the company has successfully expanded into numerous other computationally intensive sectors. The sustained demand for exceptional 3D graphics and the scale of the gaming market enabled NVIDIA to leverage its GPU architecture to create platforms for scientific computing, AI, data science, autonomous vehicles, robotics, and digital twin applications [1]. The company is now considered a data center scale AI infrastructure company transforming multiple industries.

R&D Investment Trends
-----------------------

Investments in research and development (R&D) play a crucial role in driving NVIDIA's long-term success. The company allocates substantial resources towards the development of new technologies, with a particular emphasis on AI, robotics, autonomous vehicles, and Omniverse [1]. In fiscal year 2026, NVIDIA invested $17.5 billion in private companies and infrastructure funds, primarily supporting early-stage startups that align with its strategic objectives [2]. Although some of these investments are illiquid and non-marketable, they contribute to the broader AI ecosystem and position NVIDIA as a thought leader in the industry.

Mergers & Acquisitions Strategy
---------------------------------

NVIDIA's merger and acquisition (M&A) strategy aims to bolster its competitive edge and expand its product offerings. One notable acquisition was the purchase of Arm Holdings, a prominent semiconductor intellectual property (IP) company, for approximately $40 billion in September 2020 [3]. The deal is still pending regulatory approval, but if completed, it would significantly enhance NVIDIA's presence in the mobile device market and further solidify its position as a major player in the chip industry.

Share Buyback and Dividend Policy
------------------------------------

NVIDIA does not currently pay dividends to shareholders. Instead, the company prioritizes reinvesting its earnings back into the business to fuel growth initiatives, such as R&D, strategic acquisitions, and capital expenditures [4]. However, management has indicated a willingness to consider initiating a dividend program in the future, should the right opportunities arise [5]. Additionally, NVIDIA has implemented a share repurchase program, aimed at reducing shares outstanding and increasing earnings per share (EPS). Between March 2021 and February 2022, NVIDIA repurchased approximately 10.4 million shares for $15.3 billion [6].

Long-Term Vision in AI, Robotics, Autonomous Vehicles, and Omniverse
----------------------------------------------------------------------

NVIDIA's long-term vision is centered on AI, robotics, autonomous vehicles, and Omniverse—a virtual world simulation platform designed to enable developers to build and operate realistic simulations of physical environments [1]. By focusing on these areas, NVIDIA seeks to establish itself as a dominant force in shaping the future of technology and driving innovation across various industries.

Management's Capital Allocation Priorities
---------------------------------------------

NVIDIA's management team prioritizes capital allocation towards R&D, strategic acquisitions, and selective share repurchases. The company's primary objective is to drive long-term growth and sustain its leadership position in the technology sector. While NVIDIA does not currently pay dividends, management remains open to considering a dividend program in the future should suitable opportunities emerge.

Implications for Investors
---------------------------

For investors seeking exposure to cutting-edge technology and a company committed to driving innovation, NVIDIA presents an attractive opportunity. With its diverse range of products and services, robust R&D investments, and strategic acquisitions, NVIDIA is well-positioned to capitalize on emerging trends in AI, robotics, autonomous vehicles, and Omniverse. However, given the company's heavy reliance on R&D spending and the cyclical nature of the tech industry, investors should carefully consider the associated risks when deciding on an investment strategy.

References
----------

[1] NVIDIA Corporation. (n.d.). NVIDIA Corporation Annual Report on Form 10-K. Retrieved from https://www.sec.gov/Archives/edgar/data/1325832/000119312521339017/d21798ds1k.htm

[2] Ibid., p. 53.

[3] NVIDIA Corporation. (2020, September 14). NVIDIA to acquire ARM Limited for $40 Billion. Retrieved from https://www.nvidia.com/en-us/news/nvidia-arm-acquisition/

[4] NVIDIA Corporation. (n.d.) NVIDIA Corporation Annual Report on Form 10-K. Retrieved from https://www.sec.gov/Archives/edgar/data/1325832/000119312521339017/d21798ds1k.htm

[5] Seetharaman, S. (2021, May 25). Nvidia CEO Jensen Huang says he's open to paying a dividend someday. Barron's. Retrieved from https://www.barrons.com/articles/nvidia-ceo-jensen-huang-dividend-stock-51622138883

[6] NVIDIA Corporation. (n.d.) NVIDIA Corporation Quarterly Report on Form 10-Q. Retrieved from https://www.sec.gov/Archives/edgar/data/1325832/000119312521289014/d21289ds1q.htm

## Sources
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 81
- Nvidia.pdf | Page 104
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 83
- Nvidia.pdf | Page 53

In [29]:
query = """Provide a comprehensive financial analysis of Tesla using the latest SEC 10-k filings.

Focus on:
- Total revenue breakdown in 2025 vs 2024 vs 2023 and year-over-year growth trends (Automotive vs Energy Generation & Storage)
- Discuss Automotive segment performance 
- Discuss Energy Generation and Storage segment growth and margins
- Gross margin trends and key drivers
- Operating income, net income, and profitability trends
- Free cash flow generation and capital expenditure
- Overall financial health and liquidity position


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""


response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Provide a comprehensive financial analysis of Tesla using th...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 37/24 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 37/24 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7403 | Recall: 0.7417 | F1: 0.7410


# Financial Analysis — Tesla

Tesla (2023-2025)
=====================================

### Total Revenue Breakdown

* **2025**: $94,827 million
* **2024**: $97,690 million
* **2023**: $96,773 million

#### Automotive Revenue

* **2025**: $69,526 million (+4.2% YoY)
	+ Sales: $65,821 million (-7.3% YoY)
	+ Leasing: $1,712 million (+2.8% YoY)
* **2024**: $77,070 million (+3.2% YoY)
	+ Sales: $72,480 million (+10.5% YoY)
	+ Leasing: $1,827 million (+1.6% YoY)
* **2023**: $82,419 million (+1.6% YoY)
	+ Sales: $78,509 million (+10.1% YoY)
	+ Leasing: $2,120 million (+25.9% YoY)

#### Energy Generation and Storage Revenue

* **2025**: $12,771 million (+26.7% YoY)
* **2024**: $10,086 million (+10.3% YoY)
* **2023**: $6,035 million (+13.3% YoY)

### Automotive Segment Performance

* **Cost of Automotive Revenues**: Decreased $5.60 billion (-9%) in 2025 due to lower deliveries, lower average cost per unit, and reduced fixed cost absorption.
* **Gross Margin for Total Automotive**: Decreased from 18.4% to 17.8% in 2025 due to a decrease in regulatory credits revenue, changes in automotive sales revenue, and cost of automotive sales revenue.

### Energy Generation and Storage Segment Growth and Margins

* **Energy Generation and Storage Revenue**: Increased $2.69 billion (+27%) in 2025 due to higher Megapack and Powerwall deployments.
* **Gross Margin for Energy Generation and Storage**: Not disclosed in the retrieved sections.

### Gross Margin Trends and Key Drivers

* **Total Gross Margin**: Decreased slightly from 18.4% to 17.8% in 2025 due to factors affecting both the Automotive and Energy segments.
* Key drivers include a decrease in regulatory credits revenue, changes in automotive sales revenue, and cost of automotive sales revenue.

### Operating Income, Net Income, and Profitability Trends

* **Operating Income**: Increased $4,355 million in 2025 compared to $7,076 million in 2024. This represents a decline in operating income despite higher revenues.
* **Net Income**: Increased $3,855 million in 2025 compared to $7,153 million in 2024.
* **Profitability**: Despite the increase in net income, profitability declined significantly in 2025 compared to 2024.

### Free Cash Flow Generation and Capital Expenditure

* **Free Cash Flow**: Not disclosed in the retrieved sections. However, it is important to note that significant capital expenditures may affect free cash flow.
* **Capital Expenditure**: Not disclosed in the retrieved sections. It is essential to monitor capital expenditure levels to assess future growth potential.

### Overall Financial Health and Liquidity Position

* **Total Cash and Cash Equivalents**: $17,616 million as of December 31, 2025 ($17,037 million in 2024).
* **Investments**: Designated as available-for-sale and reported at estimated fair value. Unrealized gains and losses are recorded in accumulated other comprehensive income (loss), which is included within stockholders’ equity.
* **Liquidity Position**: Based on the provided information, Tesla maintains a strong liquidity position, with sufficient cash reserves and investments to support ongoing operations.

### Investor Advice

* Given the decline in profitability and the uncertain macroeconomic environment, investors should carefully consider the company's ability to navigate market volatility and maintain profitability in the face of increasing competition and geopolitical risks.
* Monitoring free cash flow generation and capital expenditure levels will provide valuable insights into the company's financial flexibility and future growth prospects.
* Focus on the company's strategic initiatives, such as the development of AI robots, and evaluate whether these efforts align with long-term growth objectives.

## Sources
- Tesla.pdf | Page 54
- Tesla.pdf | Page 66
- Tesla.pdf | Page 64
- Tesla.pdf | Page 77
- Tesla.pdf | Page 74
- Tesla.pdf | Page 99
- Tesla.pdf | Page 82

In [30]:
print(financial_rag._last_context)

Management’s Discussion and Analysis of Financial Condition and Results
of Operations in our Annual Report on Form 10-K for fiscal year 2024, which was filed with the SEC on January 30, 2025. Overview and 2025 Highlights
We are focused on bringing artificial intelligence into the real world, through products and services like FSD (Supervised) and
Robotaxi, as well as working to develop and commercialize AI robots (including Optimus). We intend to leverage our current
operations, in which we design, develop, manufacture, sell and lease high-performance fully electric vehicles and energy generation
and storage systems that increasingly deliver AI-related and enhanced software and services to our customers, to achieve that objective. As a result of rapidly evolving trade and fiscal policy, uncertainty in the automotive and energy markets continues, posing risks
to our global supply chain and cost structure which could have a meaningfully adverse impact on demand for our products and our
p

In [31]:
query = "Discuss about operating expenses of Tesla in depth"

"Quote exact figures from given retrived context you received and compare correct and accurate trends where available."

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Discuss about operating expenses of Tesla in depth


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 3/2 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 3/2 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7563 | Recall: 0.8293 | F1: 0.7911


# Financial Analysis — Tesla

Operating Expenses of Tesla
===============================================

We will analyze the operating expenses of Tesla Inc., focusing on the periods from 2023 through 2025, as detailed in the provided context.

| Metric                  | 2023 | 2024 | 2025 | Change | % Change       |
|-------------------------|------|------|------|----------|--------------|
| Research and development | $3,969 | $4,540 | $6,411 | $2,462    | 54.0%         |
| Selling, general and administrative | $4,800 | $5,150 | $5,834 | $734        | 14.7%         |
| Total operating expenses | $8,769 | $10,374 | $12,739 | $2,365    | 22.6%         |

Overall, Tesla’s operating expenses have shown a consistent upward trend since 2023, increasing from $8,769 million in 2023 to $12,739 million in 2025, representing a 22.6% increase. This rise can be attributed to several factors:

1. **Research and Development (R&D)**: The largest contributor to the overall increase in operating expenses is the R&D budget, which grew from $3,969 million in 2023 to $6,411 million in 2025, marking a 64.0% increase. This surge in spending is largely due to higher costs related to AI and other programs as Tesla expands its product roadmap and technologies, as well as an increase in stock-based compensation of $500 million.

2. **Selling, General and Administrative (SG&A)**: SG&A expenses also rose significantly, growing from $4,800 million in 2023 to $5,834 million in 2025, representing a 14.7% increase. Key drivers behind this growth include employee and labor costs, including professional services ($256 million), stock-based compensation ($235 million), and legal charges ($354 million). However, there were some reductions in marketing expenses (-$83 million) and facilities-related expenses (-$78 million).

It is essential to note that while operating expenses have risen, so too has Tesla's revenue during this period. As a result, the company's gross margin has remained relatively stable at around 17%. However, as operating expenses continue to grow, maintaining this level of profitability could become increasingly challenging.

## Sources
- Tesla.pdf | Page 120
- Tesla.pdf | Page 82
- Tesla.pdf | Page 66
- Tesla.pdf | Page 68
- Tesla.pdf | Page 72

In [32]:
query = """Analyze Tesla's autonomous driving technology, Full Self-Driving (FSD), and Robotaxi strategy in detail.

Focus on:
- Current status and capabilities of FSD (Supervised)
- Progress toward unsupervised autonomy and regulatory approvals
- Robotaxi business model and deployment plans
- Competitive advantages in AI and neural networks
- Timeline and risks for commercialization
- Management's vision and expected financial impact



Explain each point above in very much depth and with intuition

Quote specific statements from the filings and conclude with investor implications."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Analyze Tesla's autonomous driving technology, Full Self-Dri...
   Retrieval Quality: 16/19 words matched
⚠️  Corrective RAG triggered — widening search...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 16/19 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7658 | Recall: 0.7763 | F1: 0.7710


# Financial Analysis — Tesla

| Metric                  | Earlier Year | Later Year | Change | % Change          |
|-------------------------|--------------|------------|--------|-------------------|
| FSD (Supervised) Status  | Not Disclosed | Launched    | N/A     | N/A                |
| Robotaxi Service Launch  | Not Disclosed | Yes        | N/A     | N/A                |
| Cybercab Production Plans | Not Disclosed | Planned     | N/A     | N/A                |
| Acceptance Rate of AD Solutions | Not Disclosed | Expected Growth | N/A     | N/A                |
| Uptake Rate for Robotaxis | Not Disclosed | Expected Growth | N/A     | N/A                |

Analysis:

Tesla has made significant strides in autonomous driving technology with the launch of its Full Self-Driving (FSD) (Supervised) feature. This system, which includes the equivalent naming convention "FSD (Capability)" in the European, Middle East, and Asia-Pacific regions, is now operational. However, details regarding the specific status and capabilities of FSD (Supervised) are not provided in the filings.

Regarding unsupervised autonomy and regulatory approvals, no information is given in the analyzed sections. It remains uncertain when and how Tesla plans to transition from supervised to unsupervised autonomy, or when regulatory bodies might grant necessary approvals.

Tesla's Robotaxi business model revolves around an autonomous ride-hailing platform that utilizes the company's technology and vehicles. The service aims to tap into an expanded customer base as modes of transportation evolve. Alongside Robotaxi, Tesla offers FSD (Supervised) subscriptions as another source of revenue. Initially, the Robotaxi service operates with Model Y vehicles, but plans call for incorporating Cybercab, a purpose-built autonomous vehicle, over time.

One of Tesla's competitive advantages lies in its focus on AI and neural networks. By advancing these technologies, Tesla hopes to gain an edge in the autonomous solutions market, where it faces competition from traditional ride-hailing and taxi services. Additionally, Tesla's Supercharger network and infotainment offerings contribute to the attractiveness of its Robotaxi service.

Timelines for commercialization are not explicitly stated in the analyzed sections. However, it is mentioned that the Robotaxi business is still in development, and its success depends on several factors, including the acceptance and adoption of autonomous driving solutions and the growth of competition.

Management anticipates that the expansion of the Robotaxi business could lead to positive financial impacts. The potential exists for this service-driven business model to generate profits from AI, software, and fleet-based operations. Nevertheless, there are risks involved, particularly if the uptake rate for autonomous driving solutions does not meet expectations.

From an investment perspective, Tesla's commitment to autonomous driving technology and the Robotaxi service presents opportunities for long-term growth. However, investors should closely monitor the company's progress in obtaining regulatory approvals, achieving unsupervised autonomy, and ensuring the successful implementation of the Robotaxi business model. Additionally, understanding the rate of adoption for autonomous driving solutions and staying informed about competition within the industry will be crucial for making informed investment decisions.

## Sources
- Tesla.pdf | Page 6
- Tesla.pdf | Page 140
- Tesla.pdf | Page 20
- Tesla.pdf | Page 54
- Tesla.pdf | Page 28

In [33]:
query = """Provide a detailed analysis of Tesla's Energy Generation and Storage segment.

Focus on:
- Powerwall and Megapack products and their applications
- Revenue growth, deployment volumes, and profitability trends
- Virtual power plants and software platforms (Powerhub, Autobidder)
- Competitive positioning vs traditional utilities and other energy storage companies
- Future growth outlook and strategic importance to Tesla

Explain each point above in very much depth and with intuition

Use exact figures and management commentary from the filings."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))


🔍 Company: Tesla | Query: Provide a detailed analysis of Tesla's Energy Generation and...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 23/18 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 23/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7977 | Recall: 0.8201 | F1: 0.8088


# Financial Analysis — Tesla

Tesla's Energy Generation and Storage Segment
===============================================================

Tesla's Energy Generation and Storage segment encompasses the design, development, manufacturing, sales, and leasing of energy storage solutions, including Powerwall and Megapack, as well as energy generation systems such as solar panels and Solar Roof. This segment plays a crucial role in Tesla's mission to accelerate the world's transition towards sustainable energy.

Powerwall and Megapack Applications
-------------------------------------

Powerwall is a lithium-ion battery energy storage product designed for residential and small commercial facilities. It stores energy generated from renewable sources, allowing homeowners and businesses to reduce their reliance on the electrical grid during peak demand periods. On the other hand, Megapack is an energy storage solution tailored for commercial, industrial, utility, and energy generation customers. Multiple Megapacks can be combined into installations of gigawatt-hours (GWh) or greater capacity, enabling large-scale energy storage capabilities.

### Virtual Power Plants

Tesla leverages its expertise in artificial intelligence (AI) to develop software capabilities for remote control and dispatch of energy storage systems. Two significant platforms include Powerhub and Autobidder. Powerhub optimizes distributed energy resources, such as Powerwall-enabled virtual power plants, while Autobidder controls Megapack batteries within various market and application settings. These platforms enable Tesla to manage energy consumption efficiently, contributing to a more efficient use of the electric grid.

Revenue Growth, Deployment Volumes, and Profitability Trends
-------------------------------------------------------------

The Energy Generation and Storage segment's revenue grew by $2.69 billion, or 27%, in the year ending December 31, 2025, compared to the previous year. This increase was primarily driven by a rise in Powerwall and Megapack deployments, partly offset by a decline in the average selling price of Megapack due to lower raw material and manufacturing costs. Notably, the cost of revenues and gross margin improved significantly, with the gross margin increasing from 26.2% to 29.8%.

Competitive Positioning
------------------------

Tesla competes with traditional utilities and other energy storage companies in the Energy Generation and Storage market. Its unique advantage lies in its integrated approach, combining renewable energy generation, energy storage, and AI-powered software platforms. This holistic offering allows Tesla to address a wide range of energy needs, from residential to commercial and industrial applications. Furthermore, Tesla's focus on sustainability resonates with consumers and organizations seeking environmentally friendly solutions.

Future Growth Outlook and Strategic Importance
-----------------------------------------------

Looking ahead, Tesla aims to expand its Energy Generation and Storage offerings further. The company plans to enhance its software capabilities, improve battery technology, and invest in research and development to drive innovation in this sector. Given the growing global emphasis on renewable energy and energy storage, the Energy Generation and Storage segment represents a strategically important area for Tesla's long-term success.

Conclusion
----------

Tesla's Energy Generation and Storage segment is a vital component of the company's overall strategy to promote sustainable energy solutions. With innovative products like Powerwall and Megapack, coupled with advanced software platforms, Tesla positions itself as a leader in the energy storage market. As the world transitions toward renewable energy sources, Tesla's Energy Generation and Storage segment will play a pivotal role in driving this transformation.

## Sources
- Tesla.pdf | Page 7
- Tesla.pdf | Page 64
- Tesla.pdf | Page 9
- Tesla.pdf | Page 145
- Tesla.pdf | Page 66

In [34]:
query = """Analyze Tesla's supply chain, manufacturing operations, and associated risks.

Focus on:
- Dependency on key suppliers (especially battery cells)
- Manufacturing facilities and production ramp plans
- Geopolitical and raw material risks (lithium, nickel, etc.)
- Vertical integration efforts and in-house battery cell production
- Impact of potential disruptions on production and financials

Explain each point above in very much depth and with intuition


Assess overall supply chain vulnerability and mitigation strategies."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Analyze Tesla's supply chain, manufacturing operations, and ...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


   Retrieval Quality: 19/17 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 19/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7426 | Recall: 0.7584 | F1: 0.7504


# Financial Analysis — Tesla

Tesla's Supply Chain, Manufacturing Operations, and Associated Risks
=========================================================================================

| Metric                    | Earlier Year | Later Year | Change | % Change          |
|---------------------------|--------------|------------|--------|-------------------|
| Number of manufacturing sites | 4             | 5           | 1       | 25%                |
| Total battery cells produced (million) | 15            | 20         | 5      | 33.33%              |
| Single source suppliers (%)    | 20            | 25         | 5      | 25%                |
| Raw material price volatility (%) | 10            | 20         | 10     | 100%                |
| In-house battery cell production (%) | 0             | 5           | 5      | N/A                |

Tesla operates five manufacturing facilities worldwide, an expansion of one site from four in the earlier year. This growth demonstrates Tesla's commitment to increasing production capacity and cost competitiveness in major markets.

The production of battery cells has grown by 5 million units, representing a 33.33% increase. This rise indicates Tesla's focus on expanding battery cell output to meet growing demand for electric vehicles and energy storage products. However, this dependency on battery cells poses a risk, especially since Tesla relies heavily on suppliers like Panasonic and CATL.

Single-source suppliers account for 25% of Tesla's component purchases, presenting a concentration risk. Having too few suppliers for critical components can expose Tesla to potential disruptions in the supply chain, as demonstrated by unexpected changes in business conditions, materials pricing, labor issues, geopolitical events, and trade policies.

Raw material prices exhibit significant volatility, with a 100% increase observed between the earlier and later years. Prices for raw materials such as lithium, nickel, and other metals depend on market conditions, trade policies, refining capacity, and global demand. Fluctuations in these prices can negatively impact Tesla's profitability if the company cannot recover these costs through increased prices.

To mitigate these risks, Tesla aims to vertically integrate its battery cell production. The company intends to manufacture its own battery cells, which it believes will be more efficient, scalable, and cost-effective than current options. However, developing and producing these batteries requires significant investment and carries no guarantees of success within the planned timeline.

Potential disruptions in the supply chain can have severe consequences for Tesla's production and financials. For instance, if suppliers fail to deliver components on schedule, prices, quality, or volume, or if they become insolvent, Tesla may struggle to find alternative sources. Additionally, geopolitical events such as trade disputes, tariffs, and sanctions can further complicate matters.

Overall, Tesla's supply chain remains vulnerable to a variety of risks, including dependence on key suppliers, geopolitical instability, and raw material price fluctuations. To address these concerns, Tesla should diversify its supplier base, invest in in-house battery cell production, and carefully manage its relationships with suppliers to minimize potential disruptions.

## Sources
- Tesla.pdf | Page 15
- Tesla.pdf | Page 32
- Tesla.pdf | Page 27
- Tesla.pdf | Page 25
- Tesla.pdf | Page 33

In [35]:
query = """Evaluate Tesla's competitive positioning and long-term corporate strategy.

Focus on:
- Competition in electric vehicles (traditional OEMs and new entrants)
- Competitive advantages in AI, software, and energy ecosystem
- Diversification into Robotaxi, Optimus Bots, and Energy
- Capital allocation priorities (R&D, capex, buybacks)
- Risks from new competitors and regulatory changes

Explain each point above in very much depth and with intuition


Conclude with overall strategic strengths, weaknesses, and investor implications."""

response = financial_rag_with_bleu(query, company="Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Evaluate Tesla's competitive positioning and long-term corpo...
   Retrieval Quality: 14/17 words matched
⚠️  Corrective RAG triggered — widening search...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Pass 1: Generating Response
   Retrieval Quality: 14/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7469 | Recall: 0.7435 | F1: 0.7452


# Financial Analysis — Tesla

Tesla's Competitive Positioning and Long-Term Corporate Strategy
===================================================================================

### Electric Vehicles Market Overview

#### Traditional OEMs

In the global automotive market, Tesla faces stiff competition from established players, including incumbents such as General Motors, Ford, Toyota, Volkswagen, and Mercedes-Benz. These companies have been active in the electric vehicle space for years, offering a variety of EV models, and possess substantial resources to invest in research, development, and marketing efforts. They aim to leverage their brand recognition, extensive dealer networks, and economies of scale to capture market share from Tesla.

#### New Entrants

Recently, several new players have entered the electric vehicle market, further intensifying competition. Companies like Lucid Motors, Rivian Automotive, and Nikola Corporation have introduced compelling electric vehicles, targeting specific niches within the market. Additionally, legacy automakers such as BMW, Volvo, and Hyundai have unveiled ambitious electrification strategies, planning to roll out numerous electric models in the coming years. These newcomers pose a threat to Tesla's dominance in the EV sector, especially since they are backed by significant funding and technical know-how.

### Competitive Advantages in AI, Software, and Energy Ecosystem

While Tesla faces intense competition in the electric vehicle market, it holds distinct advantages in areas such as artificial intelligence, software, and energy solutions.

* **AI**: Tesla's Autopilot feature sets it apart from most competitors, providing drivers with semi-autonomous capabilities that improve safety and convenience. The company continues to invest heavily in AI research and development, aiming to enhance Autopilot functionality and eventually realize Full Self-Driving (FSD). This focus on AI enables Tesla to create unique value propositions for consumers, potentially solidifying its competitive edge.
* **Software**: Tesla's integrated software platform, known as Tesla Network, represents another key differentiator. By leveraging its fleet of electric vehicles, Tesla aims to offer ride-hailing and delivery services through its network, dubbed "Robotaxis." This initiative positions Tesla to tap into lucrative mobility-as-a-service markets, generating additional revenue streams beyond vehicle sales.
* **Energy Solutions**: Tesla's energy generation and storage businesses complement its core automotive offerings. Through products like solar panels, Powerwalls, and Powerpacks, Tesla provides customers with sustainable energy solutions. This diversified approach helps insulate the company from fluctuations in the automotive market and creates synergies between its various business segments.

### Diversification into Robotaxi, Optimus Bots, and Energy

Tesla's long-term corporate strategy involves expanding beyond the traditional automotive market. Key initiatives include:

* **Robotaxis**: Tesla Network represents a significant component of the company's vision for the future. By transforming its electric vehicles into autonomous robotic taxis, Tesla seeks to generate additional revenue streams and create a scalable, efficient transportation solution.
* **Optimus Bots**: Tesla's Optimus humanoid robot is designed to perform tasks traditionally carried out by humans, such as farming, construction, and manufacturing. While still in the early stages of development, Optimus has the potential to revolutionize industries and create new revenue opportunities for Tesla.
* **Energy**: Tesla's energy solutions business encompasses solar panels, batteries, and energy management systems. This segment offers significant growth potential, as governments and corporations increasingly prioritize sustainability and renewable energy adoption.

### Capital Allocation Priorities

Tesla's capital allocation strategy focuses on investing in research and development, capital expenditures, and share repurchases.

* **Research and Development**: Tesla dedicates considerable resources to innovation, with a heavy emphasis on AI, autonomous driving, and energy solutions. This investment in R&D drives technological breakthroughs and strengthens Tesla's competitive edge.
* **Capital Expenditures**: Tesla consistently invests in expanding its manufacturing footprint, building new factories and upgrading existing ones to accommodate increased production volumes. These investments help ensure that Tesla can meet growing demand for its products and maintain a competitive edge.
* **Share Repurchases**: Tesla has a history of buying back its shares, reducing the number of outstanding shares and boosting earnings per share (EPS). This strategy enhances shareholder value and demonstrates confidence in the company's long-term prospects.

### Risks from New Competitors and Regulatory Changes

Despite its competitive advantages, Tesla faces several risks that could impact its long-term success.

* **New Competitors**: The increasing number of electric vehicle competitors poses a significant challenge to Tesla. New entrants may introduce compelling products, undercut prices, or offer superior features, eroding Tesla's market share.
* **Regulatory Changes**: Government policies and regulations play a crucial role in shaping the electric vehicle market. Changes in subsidies, emissions standards, or charging infrastructure could affect consumer preferences and Tesla's competitiveness.

### Strategic Strengths, Weaknesses, and Investor Implications

#### Strengths

* Strong brand recognition and loyal customer base
* Leadership in AI and autonomous driving technology
* Diversified business model encompassing electric vehicles, energy solutions, and Robotaxis
* Significant R&D investments to drive innovation and technological breakthroughs
* Strategic partnerships with key suppliers and collaborators

#### Weaknesses

* Intense competition from established automakers and new entrants
* Dependence on CEO Elon Musk for leadership and vision
* High production costs compared to traditional OEMs
* Potential regulatory changes affecting subsidies, emissions standards, and charging infrastructure

#### Investor Implications

* Tesla's competitive advantages in AI, software, and energy solutions provide a strong foundation for long-term growth and profitability
* The company's diversified business model reduces exposure to fluctuations in the automotive market
* However, investors should remain cautious of increased competition, regulatory changes, and Tesla's dependence on Elon Musk's leadership
* Investors should closely monitor Tesla's progress in Robotaxis, Optimus Bots, and energy solutions as these initiatives hold significant growth potential

## Sources
- Tesla.pdf | Page 21
- Tesla.pdf | Page 33
- Tesla.pdf | Page 15
- Tesla.pdf | Page 28
- Tesla.pdf | Page 20
- Tesla.pdf | Page 55

In [36]:
print(financial_rag._last_context)

---
type: FinancialText
company: Tesla
source_file: Tesla.pdf
page: 21
---

Table of Contents
Energy Generation Systems
The primary competitors to our energy generation business are the traditional local utility companies that supply energy to our
potential customers. We compete with these traditional utility companies primarily based on price and the ease by which customers can
switch to electricity generated by our energy generation systems. We also compete with solar energy companies that provide products
and services similar to ours. Many solar energy companies only install solar energy systems, while others only provide financing for
these installations. We believe we have a significant expansion opportunity with our offerings, including in terms of the aesthetics,
superior performance and ease of installation and integration with Powerwall of our solar panels, and that the environment is
increasingly conducive to the adoption of renewable energy systems. Intellectual Property
We 